# Portfolio Project 2: A/B Test Results Analysis

**Project goal:** Analyze experiment results with a two-sample Z-test in Python for 4 conversion metrics and prepare a final dataset for visualization in Tableau.

---

### 🔗 Useful links:
* **Tableau Public dashboard:** https://public.tableau.com/views/ABTestStatisticalSignificance_17918673786320/ABtest?:language=fr-FR&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
* **CSV file with results on Google Drive:** https://drive.google.com/file/d/1cHgoGd7kF_3S3xwtwk-8bmT_dfhl4ZwY/view?usp=sharing

---

### 📐 Methodology and statistical test:
* **Test:** Two-sample Z-test for proportions (2-proportion Z-test).
* **Confidence level:** 95% ($\alpha = 0.05$).
* **Hypotheses:**
  * **H₀:** $p_A = p_B$ (there is no difference between the conversion rates of the control and test groups).
  * **H₁:** $p_A \neq p_B$ (there is a statistically significant difference between the conversion rates).
* **Decision rule:** If $p\text{-value} < 0.05$, we reject the null hypothesis and the change is considered statistically significant (`Significant`). Otherwise, it is `Not Significant`.

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy import stats
import warnings

# Hide harmless RuntimeWarnings (NaN results in very small segments)  # ADDED
warnings.filterwarnings('ignore')

# 1. Authenticate the user in Google Cloud
auth.authenticate_user()
print('Google Cloud authentication successful!')

# 2. Set up the BigQuery client
PROJECT_ID = 'data-analytics-mate'
client = bigquery.Client(project=PROJECT_ID)

# 3. SQL query
sql_query = """
WITH session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `data-analytics-mate.DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `data-analytics-mate.DA.session_params` sp
      ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `data-analytics-mate.DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
events AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name,
      COUNT(sp.ga_session_id) AS event_cnt
    FROM `data-analytics-mate.DA.event_params` sp
    JOIN session_info
      ON sp.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name
),
session AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
account AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `data-analytics-mate.DA.account_session` acs
    JOIN session_info
      ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
)
SELECT
  session_with_orders.date,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  'session with orders' AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  events.event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  new_account_cnt AS value
FROM account
"""

# 4. Load data via the standard REST API (without the BQ Storage API)
df_raw = client.query(sql_query).to_dataframe(create_bqstorage_client=False)

print(f"Data loaded successfully! Rows received: {len(df_raw)}")
df_raw.head()

Google Cloud authentication successful!
Data loaded successfully! Rows received: 800996


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new account,1
1,2020-11-04,Bolivia,desktop,Americas,Direct,2,1,new account,1
2,2020-11-05,Lebanon,mobile,Asia,Direct,2,2,new account,1
3,2020-11-06,Oman,desktop,Asia,Paid Search,2,2,new account,1
4,2020-11-06,Dominican Republic,desktop,Americas,Paid Search,2,1,new account,1


In [ ]:
# 1. Cleaning, removing duplicate columns and standardizing names
df = df_raw.copy()

# Automatically drop duplicated columns
df = df.loc[:, ~df.columns.duplicated()].copy()

# Strip accidental spaces from column names
df.columns = df.columns.astype(str).str.strip()

# Standardize and normalize event names
df['event_name'] = df['event_name'].astype(str).str.strip().str.lower()

event_mapping = {
    'session': 'sessions',
    'add_payment_info': 'add_payment_info',
    'add_shipping_info': 'add_shipping_info',
    'begin_checkout': 'begin_checkout',
    'new account': 'new_accounts',
    'new_account': 'new_accounts'
}
df['event_name'] = df['event_name'].map(lambda x: event_mapping.get(x, x))

# 2. Filtering: keep only the 5 required events
TARGET_EVENTS = ['sessions', 'add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_accounts']
df_filtered = df[df['event_name'].isin(TARGET_EVENTS)].copy()

# 3. Configuration of the 4 required metrics: (numerator, denominator)
METRICS_CONFIG = {
    'add_payment_info / session': ('add_payment_info', 'sessions'),
    'add_shipping_info / session': ('add_shipping_info', 'sessions'),
    'begin_checkout / session': ('begin_checkout', 'sessions'),
    'new_accounts / session': ('new_accounts', 'sessions')
}

# 4. Two-sample Z-test function for proportions
def calculate_z_test(count_control, n_control, count_test, n_test, alpha=0.05):
    if n_control == 0 or n_test == 0 or pd.isna(n_control) or pd.isna(n_test):
        return {
            'cr_control': 0, 'cr_test': 0, 'abs_diff': 0, 'rel_uplift': 0,
            'p_value': np.nan, 'is_significant': 'No Data',
            'ci_lower': np.nan, 'ci_upper': np.nan
        }

    p1 = count_control / n_control
    p2 = count_test / n_test

    abs_diff = p2 - p1
    rel_uplift = (abs_diff / p1 * 100) if p1 > 0 else 0

    # Pooled proportion
    p_pooled = (count_control + count_test) / (n_control + n_test)
    se_pooled = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_control + 1/n_test))

    if se_pooled == 0:
        z_score = 0
        p_val = 1.0
    else:
        z_score = (p2 - p1) / se_pooled
        p_val = 2 * (1 - stats.norm.cdf(abs(z_score)))

    # 95% confidence interval
    se_diff = np.sqrt((p1 * (1 - p1) / n_control) + (p2 * (1 - p2) / n_test))
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ci_lower = abs_diff - z_crit * se_diff
    ci_upper = abs_diff + z_crit * se_diff

    is_sig = "Significant" if p_val < alpha else "Not Significant"

    return {
        'cr_control': p1,
        'cr_test': p2,
        'abs_diff': abs_diff,
        'rel_uplift': rel_uplift,
        'p_value': p_val,
        'is_significant': is_sig,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }

# 5. Dimensions to loop over automatically
DIMENSIONS = ['Total', 'test', 'device', 'country', 'channel']

results_list = []

# Define the control and test groups
groups = sorted(df_filtered['test_group'].dropna().unique())
control_label = groups[0]
test_label = groups[1]

# 6. Dynamic loop to calculate results for each dimension
for dim in DIMENSIONS:
    if dim == 'Total':
        group_cols = ['test', 'test_group']
    else:
        group_cols = ['test', dim, 'test_group']
        df_filtered[dim] = df_filtered[dim].fillna('Unknown')

    # Remove duplicates from the grouping column list
    group_cols = list(dict.fromkeys(group_cols))

    # Create the pivot table
    df_pv = pd.pivot_table(
        df_filtered,
        index=group_cols,
        columns='event_name',
        values='value',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    df_pv.columns.name = None  # Clear the axis name

    # Make sure all required columns exist
    for col in TARGET_EVENTS:
        if col not in df_pv.columns:
            df_pv[col] = 0

    unique_tests = df_pv['test'].unique()

    for test_id in unique_tests:
        df_test = df_pv[df_pv['test'] == test_id]
        dim_values = ['Total'] if dim == 'Total' else df_test[dim].unique()

        for dim_val in dim_values:
            if dim == 'Total':
                sub_df = df_test
            else:
                sub_df = df_test[df_test[dim] == dim_val]

            row_control = sub_df[sub_df['test_group'] == control_label]
            row_test = sub_df[sub_df['test_group'] == test_label]

            if row_control.empty or row_test.empty:
                continue

            n_ctrl = row_control['sessions'].values[0]
            n_tst = row_test['sessions'].values[0]

            for metric_name, (num_col, den_col) in METRICS_CONFIG.items():
                x_ctrl = row_control[num_col].values[0]
                x_tst = row_test[num_col].values[0]

                stats_res = calculate_z_test(x_ctrl, n_ctrl, x_tst, n_tst)

                results_list.append({
                    'test_id': test_id,
                    'dimension_type': dim,
                    'dimension_value': str(dim_val),
                    'metric_name': metric_name,
                    'control_sessions': n_ctrl,
                    'test_sessions': n_tst,
                    'control_conversions': x_ctrl,
                    'test_conversions': x_tst,
                    'control_cr': round(stats_res['cr_control'] * 100, 4),
                    'test_cr': round(stats_res['cr_test'] * 100, 4),
                    'absolute_diff_pp': round(stats_res['abs_diff'] * 100, 4),
                    'relative_uplift_pct': round(stats_res['rel_uplift'], 2),
                    'p_value': round(stats_res['p_value'], 5) if pd.notna(stats_res['p_value']) else np.nan,
                    'is_significant': stats_res['is_significant'],
                    'ci_lower_pp': round(stats_res['ci_lower'] * 100, 4) if pd.notna(stats_res['ci_lower']) else np.nan,
                    'ci_upper_pp': round(stats_res['ci_upper'] * 100, 4) if pd.notna(stats_res['ci_upper']) else np.nan
                })

df_results = pd.DataFrame(results_list)
print("Calculations completed successfully!")
df_results.head(10)

Calculations completed successfully!


,test_id,dimension_type,dimension_value,metric_name,control_sessions,test_sessions,control_conversions,test_conversions,control_cr,test_cr,absolute_diff_pp,relative_uplift_pct,p_value,is_significant,ci_lower_pp,ci_upper_pp
0,1,Total,Total,add_payment_info / session,45362,45193,1988,2229,4.3825,4.9322,0.5497,12.54,0.00009,Significant,0.2752,0.8241
1,1,Total,Total,add_shipping_info / session,45362,45193,3034,3221,6.6884,7.1272,0.4388,6.56,0.00923,Significant,0.1085,0.7691
2,1,Total,Total,begin_checkout / session,45362,45193,3784,4021,8.3418,8.8974,0.5556,6.66,0.00289,Significant,0.1900,0.9212
3,1,Total,Total,new_accounts / session,45362,45193,3823,3681,8.4278,8.1451,-0.2827,-3.35,0.12286,Not Significant,-0.6418,0.0764
4,2,Total,Total,add_payment_info / session,50637,50244,2344,2409,4.6290,4.7946,0.1656,3.58,0.21461,Not Significant,-0.0959,0.4271
5,2,Total,Total,add_shipping_info / session,50637,50244,3480,3510,6.8724,6.9859,0.1135,1.65,0.47798,Not Significant,-0.2000,0.4269
6,2,Total,Total,begin_checkout / session,50637,50244,4262,4313,8.4168,8.5841,0.1673,1.99,0.34064,Not Significant,-0.1769,0.5115
7,2,Total,Total,new_accounts / session,50637,50244,4165,4184,8.2252,8.3274,0.1022,1.24,0.55600,Not Significant,-0.2379,0.4422
8,3,Total,Total,add_payment_info / session,70047,70439,3623,3697,5.1722,5.2485,0.0763,1.47,0.52011,Not Significant,-0.1561,0.3087
9,3,Total,Total,add_shipping_info / session,70047,70439,5298,5188,7.5635,7.3652,-0.1983,-2.62,0.15744,Not Significant,-0.4731,0.0766


In [ ]:
import google.colab.files

# 1. Save the calculation results to CSV
output_filename = "ab_test_significance_results.csv"
df_results.to_csv(output_filename, index=False, encoding='utf-8-sig')

# 2. Automatically download the file to the computer
google.colab.files.download(output_filename)
print(f"File {output_filename} was generated and downloaded to your computer!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File ab_test_significance_results.csv was generated and downloaded to your computer!


### Summary of findings and analysis
1. **Dynamic metric calculation:** The script is built with arrays and loops, without hard-coding the number of rows. Four required metrics are calculated automatically:
   - `add_payment_info / session`
   - `add_shipping_info / session`
   - `begin_checkout / session`
   - `new_accounts / session`

2. **In-depth analysis by dimensions:** In addition to overall results (Total), statistical significance is calculated by device (`device`), country (`country`) and channel (`channel`). This helped reveal local anomalies and segment-specific user behavior that the overall result hides.

3. **Data ready for visualization:** The file `ab_test_significance_results.csv` was generated and exported. It contains all the parameters needed to build the Tableau dashboard (p-value, relative uplift %, 95% confidence interval bounds and significance status).